# NB_reidentify · Break-Glass Re-Identification (RESTRICTED)

> ⚠️ **RESTRICTED — VAULT WORKSPACE ONLY.** This notebook reverses tokens back to the
> original values. It must live in the isolated **Vault** workspace, accessible to the
> ~2 people authorized for re-identification. Every run should be logged/audited.
>
> **SYNTHETIC DATA ONLY** in this accelerator.

Tokenization is HMAC (one-way): a token cannot be reversed by math alone. Re-identification
is only possible via a **crosswalk** table (token → original) built here — inside the Vault
workspace, which is the only place that can see both the raw `silver_*` data **and** the
Key Vault pepper. The crosswalk never leaves this workspace.

In [ ]:
from pyspark.sql import functions as F, types as T
import os, sys
# The fabric_phi_deid package may land at Files/accelerator/ OR Files/accelerator/src/
# depending on how it was uploaded. Add whichever one actually contains the package.
_CANDIDATES = [
    "/lakehouse/default/Files/accelerator",
    "/lakehouse/default/Files/accelerator/src",
]
for _p in _CANDIDATES:
    if os.path.isdir(os.path.join(_p, "fabric_phi_deid")) and _p not in sys.path:
        sys.path.append(_p)
try:
    from fabric_phi_deid.tokenization import get_pepper, tokenize  # noqa: E402
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "fabric_phi_deid not found. Upload the accelerator 'src' so that "
        "'fabric_phi_deid/' sits under Files/accelerator/ (or Files/accelerator/src/). "
        f"Searched: {_CANDIDATES}"
    ) from e

# DEMO (Option 2, synthetic only): use the SAME pepper as 02b_silver_deid, or tokens
# will NOT reverse. Paste the identical value here. Production fetches from Key Vault.
DEMO_PEPPER = "xbSJJefaA60C_s2oNjPKr3t7Z1BQaGv9Go9TH5rse-2kGAZHdVGBwA9mnItp0COf"
os.environ["PHI_DEID_PEPPER"] = DEMO_PEPPER

PEPPER = get_pepper("phi-deid-pepper")   # never print this
assert PEPPER, "Pepper required (env var for demo, Key Vault for prod)."


In [ ]:
# ============================ PARAMETERS ============================
# This notebook runs in the VAULT workspace (lh_vault attached as default).
# The two sources it needs live in OTHER workspaces and are read cross-workspace
# by fully-qualified ABFSS path (native in OneLake — no shortcuts, no copies):
#   - silver_dim_patient  (raw PHI: MRN, PatientName, DOB, ZIP)  -> PHI-Raw / lh_raw
#   - gold_safe_dim_patient (tokens, to pick a demo token)       -> PHI-Analytics / lh_analytics
# The crosswalk it BUILDS (xwalk_patient) is written to the ATTACHED lh_vault only.
RAW_WORKSPACE       = "PHI-Raw"          # holds the raw silver_* tables
RAW_LAKEHOUSE       = "lh_raw"
ANALYTICS_WORKSPACE = "PHI-Analytics"    # holds the PHI-free gold_safe_* tables
ANALYTICS_LAKEHOUSE = "lh_analytics"

RAW_ROOT = (
    f"abfss://{RAW_WORKSPACE}@onelake.dfs.fabric.microsoft.com/"
    f"{RAW_LAKEHOUSE}.Lakehouse/Tables"
)
ANALYTICS_ROOT = (
    f"abfss://{ANALYTICS_WORKSPACE}@onelake.dfs.fabric.microsoft.com/"
    f"{ANALYTICS_LAKEHOUSE}.Lakehouse/Tables"
)
print(f"Raw source (PHI):      {RAW_ROOT}")
print(f"Analytics source:      {ANALYTICS_ROOT}")
print("Crosswalk output:      attached lh_vault (Vault workspace only)")


In [ ]:
# ---------- Build the patient crosswalk (token -> original) ----------
# Recomputes the SAME keyed token used during de-identification so the crosswalk
# aligns 1:1 with gold_safe. Written to the attached Vault lakehouse only.
#
# WHY A pandas UDF (not F.udf importing fabric_phi_deid):
# Tokenization runs DISTRIBUTED on executors. A row-at-a-time UDF that imports the
# accelerator package fails on workers where the package is not on sys.path
# (TASK_WRITE_FAILED). Here the token is computed with a VECTORIZED pandas UDF over
# Python stdlib (hmac + hashlib) -- guaranteed on every node and far faster. The logic
# is byte-for-byte identical to fabric_phi_deid.tokenization.tokenize(
#   namespace="mrn", prefix="PT-", length=16); a parity guard below enforces that.
import hmac
import hashlib
import pandas as pd
from pyspark.sql.functions import pandas_udf

_MRN_NAMESPACE = "mrn"
_TOKEN_PREFIX = "PT-"
_TOKEN_LEN = 16
_UNIT_SEP = "\x1f"  # binds namespace to value so ("mrn","123") != ("mr","n123")

# Capture the pepper BY VALUE so Spark broadcasts it to executors (fetched once on the
# driver from Key Vault / env). Required for distributed keyed tokenization. Never logged.
_PEPPER_BYTES = PEPPER.encode("utf-8")


@pandas_udf(T.StringType())
def tok_mrn_udf(mrn: pd.Series) -> pd.Series:
    key = _PEPPER_BYTES

    def _tok(value):
        if value is None or value == "":
            return value
        message = f"{_MRN_NAMESPACE}{_UNIT_SEP}{value}".encode("utf-8")
        return _TOKEN_PREFIX + hmac.new(key, message, hashlib.sha256).hexdigest()[:_TOKEN_LEN]

    return mrn.astype("object").map(_tok)


# CROSS-WORKSPACE READ: raw silver_dim_patient lives in PHI-Raw / lh_raw, not the
# attached Vault lakehouse. Only the Vault can see BOTH this raw PHI and the pepper.
xwalk_patient = (
    spark.read.format("delta").load(f"{RAW_ROOT}/silver_dim_patient")
    .select("PatientKey", "MRN", "PatientName", "DateOfBirth", "ZIP")
    .withColumn("MRN", F.col("MRN").cast("string"))
    .withColumn("MRNToken", tok_mrn_udf(F.col("MRN")))
)

(xwalk_patient.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("xwalk_patient"))

# Parity guard (cheap, reads persisted table): the distributed token MUST equal the
# audited reference impl, or the crosswalk would not reverse gold_safe tokens.
_probe = (spark.table("xwalk_patient")
          .where(F.col("MRN").isNotNull())
          .select("MRN", "MRNToken").limit(1).collect())
if _probe:
    _m, _got = _probe[0]["MRN"], _probe[0]["MRNToken"]
    _ref = tokenize(_m, PEPPER, namespace="mrn", prefix="PT-")
    assert _got == _ref, (
        "Token parity FAILED: distributed token != fabric_phi_deid.tokenize. "
        "Crosswalk would not reverse gold_safe tokens."
    )

_row_count = spark.table("xwalk_patient").count()
print(f"xwalk_patient rebuilt: {_row_count:,} rows (Vault workspace only). Parity check OK.")


In [ ]:
# ---------- Break-glass lookup: token -> original identity ----------
# Example (authorized use only): resolve a single token investigators flagged.
def reidentify(mrn_token: str):
    """Return the original identity for a single MRN token. Log every call."""
    row = (spark.table("xwalk_patient")
           .where(F.col("MRNToken") == mrn_token)
           .select("PatientKey", "MRN", "PatientName", "DateOfBirth", "ZIP")
           .limit(1).collect())
    return row[0].asDict() if row else None

# Demo: pick one token from the PHI-free gold_safe layer (PHI-Analytics / lh_analytics,
# read cross-workspace) and resolve it back to the real identity via the Vault crosswalk.
sample_token = (spark.read.format("delta")
    .load(f"{ANALYTICS_ROOT}/gold_safe_dim_patient")
    .select("MRN").limit(1).collect()[0]["MRN"])
print("Resolving token:", sample_token)
print(reidentify(sample_token))


**Controls to enforce around this notebook**
- Lives in the **Vault** workspace; only re-identification approvers have access.
- Access to `xwalk_patient` is governed by OneLake security + audit logging.
- Rotating the Key Vault pepper invalidates every token → rebuild both de-id output and
  this crosswalk (the breach-recovery lever).